# Forward Polymarket data collection

## Purpose

This notebook performs **monthly forward data collection** for fixtures selected by the already-frozen Football-Data `market_maximum` home-win rule. It creates auditable league-month files for later execution and performance analysis; it does not estimate parameters, size bets, or report P&L.

The workflow is deliberately separated into two stages:

1. **Signal and match selection:** Football-Data `MaxH` and the published frozen parameters determine which fixtures qualify.
2. **Execution-data collection:** Dune is used only to identify the corresponding Polymarket home-win market and retrieve qualifying transaction observations between the Football-Data release time and kickoff.

The primary user input is `MONTH_TO_PROCESS`. The current-season Football-Data workbook should be updated manually before running a completed month.

## 1. Scientific design

For each match, the Football-Data reference probability is

$$
p_{FD} = \frac{1}{MaxH}.
$$

The frozen league parameters produce

$$
\widehat{G}(p_{FD}) = \widehat{\alpha} + \widehat{\beta}p_{FD},
\qquad
\widehat{q}_{FD} = p_{FD} + \widehat{G}(p_{FD}).
$$

A fixture is selected only when $p_{FD}$ lies inside its league's published accepted range. Polymarket prices are never passed through $\widehat{G}$ and never determine whether the Football-Data signal exists.

Football-Data's documented latest collection times define the causal retrieval window: Friday 17:00 UK for Friday–Monday fixtures and Tuesday 13:00 UK for Tuesday–Thursday fixtures. `Europe/London` is used so daylight-saving conversion is explicit.

After resolving exactly one home-win `condition_id` and YES `asset_id`, the Dune query retains the same canonical direction definitions as the private reference implementation:

- historical taker SELL / maker BUY: effective odds are `shares / amount`;
- historical taker BUY / maker SELL: effective odds are `shares / (amount + recorded fee)`.

Only observations with direction-specific effective odds at least as high as Football-Data `MaxH` are retained. This is data collection, not a claim that the displayed volume could all have been captured by a live order.

## 2. Configuration

Set `MONTH_TO_PROCESS` to the completed calendar month in `YYYY-MM` format. Credentials must be supplied through the `DUNE_API_KEY` environment variable; they must never be written into this notebook.

Finalized league-month files are protected against accidental replacement. Set `ALLOW_CORRECTION_OVERWRITE=True` only when intentionally correcting a previously collected month, and document that correction in version control.

In [ ]:
from __future__ import annotations

import sys
from datetime import time
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from football_edge.forward_polymarket import (
    DuneClient,
    collect_monthly_dune_data,
    ensure_month_is_not_finalized,
    monthly_collection_diagnostics,
    prepare_monthly_fixture_selection,
    save_monthly_outputs_by_league,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)

In [ ]:
# Main monthly input
MONTH_TO_PROCESS = "2026-09"

# Update this local workbook before collecting a completed month.
CURRENT_SEASON_FILE = PROJECT_ROOT / "data" / "holdout" / "all-euro-data-2026-2027.xlsx"
FROZEN_PARAMETERS_FILE = (
    PROJECT_ROOT
    / "docs"
    / "parameter_snapshots"
    / "market_maximum_home_win_asof_2025_26.csv"
)

TARGET_LEAGUE: str | None = None  # None processes every league in the frozen snapshot.
MARKET_OUTCOME = "home_win"
ODDS_SOURCE = "market_maximum"
FIXTURE_TIMEZONE = "Europe/London"
WEEKEND_RELEASE_TIME_UK = time(17, 0)
MIDWEEK_RELEASE_TIME_UK = time(13, 0)

DUNE_PERFORMANCE = "small"
DUNE_POLL_SECONDS = 5.0
DUNE_TIMEOUT_SECONDS = 600.0
DUNE_RESULT_LIMIT = 10_000

OUTPUT_DIR = PROJECT_ROOT / "data" / "holdout" / "dune_cache"
IDENTITY_CACHE_FILE = OUTPUT_DIR / "resolved_home_win_market_identities.csv"
SAVE_MONTHLY_OUTPUTS = True
ALLOW_CORRECTION_OVERWRITE = False
FORCE_REFRESH_MARKET_DISCOVERY = False

## 3. Frozen-rule fixture selection

The selected-month workbook is filtered to leagues present in the frozen parameter snapshot. Outcomes may be retained in the resulting audit files for later settlement, but they are not used anywhere in selection, market discovery, or transaction eligibility.

In [ ]:
selection = prepare_monthly_fixture_selection(
    CURRENT_SEASON_FILE,
    FROZEN_PARAMETERS_FILE,
    month=MONTH_TO_PROCESS,
    target_league=TARGET_LEAGUE,
    fixture_timezone=FIXTURE_TIMEZONE,
    weekend_release_time=WEEKEND_RELEASE_TIME_UK,
    midweek_release_time=MIDWEEK_RELEASE_TIME_UK,
    market_outcome=MARKET_OUTCOME,
    source=ODDS_SOURCE,
)

snapshot_metadata = ["as_of_training_end_season", "market_outcome", "source"]
print("Frozen parameter snapshot:")
for column in snapshot_metadata:
    values = selection.frozen_parameters[column].dropna().astype(str).drop_duplicates()
    print(f"  {column}: {', '.join(values)}")
display(
    selection.frozen_parameters.drop(columns=snapshot_metadata)
    .sort_values("league", kind="stable")
    .reset_index(drop=True)
    .round(6)
)

print(f"Workbook: {CURRENT_SEASON_FILE.relative_to(PROJECT_ROOT)}")
print(f"Season: {selection.season}")
print(f"Month: {selection.month}")
print(f"Football-Data matches observed in frozen-parameter leagues: {len(selection.observed_fixtures):,}")
print(f"Matches qualifying under the frozen rule: {len(selection.selected_fixtures):,}")

selection_by_league = (
    selection.observed_fixtures.groupby("league", sort=True)
    .agg(
        matches_observed=("home_team", "size"),
        matches_selected=("accepted_fd", "sum"),
        accepted_probability_min=("accepted_probability_min", "first"),
        accepted_probability_max=("accepted_probability_max", "first"),
    )
    .reset_index()
)
display(selection_by_league.round(4))

## 4. Market discovery and eligible transaction retrieval

The output guard runs before any paid Dune query. If a league-month has already been finalized, execution stops unless the explicit correction flag is enabled.

Market discovery uses normalized home/away aliases but accepts a fixture only when it resolves to one unique home-win `condition_id` and YES `asset_id`. The second query uses that exact `condition_id`; it does not repeat fuzzy team-name matching.

In [ ]:
if SAVE_MONTHLY_OUTPUTS:
    planned_files = ensure_month_is_not_finalized(
        selection.selected_fixtures,
        output_dir=OUTPUT_DIR,
        season=selection.season,
        month=selection.month,
        allow_correction_overwrite=ALLOW_CORRECTION_OVERWRITE,
        source=ODDS_SOURCE,
        market_outcome=MARKET_OUTCOME,
    )
    print(f"League-month output sets planned: {len(planned_files):,}")

dune_client = DuneClient.from_environment(
    performance=DUNE_PERFORMANCE,
    poll_seconds=DUNE_POLL_SECONDS,
    timeout_seconds=DUNE_TIMEOUT_SECONDS,
    result_limit=DUNE_RESULT_LIMIT,
)

collection = collect_monthly_dune_data(
    selection.selected_fixtures,
    client=dune_client,
    identity_cache_file=IDENTITY_CACHE_FILE,
    fixture_timezone=FIXTURE_TIMEZONE,
    force_refresh_market_discovery=FORCE_REFRESH_MARKET_DISCOVERY,
)

## 5. Monthly audit diagnostics

The six headline counts form a coverage funnel. In particular, **not discovered in trade data** is different from **resolved with no eligible execution**: the former indicates that no unique market identity was recovered, while the latter means that the market was identified but no direction-correct transaction met `MaxH` during the allowed window.

In [ ]:
diagnostics = monthly_collection_diagnostics(selection, collection)
display(diagnostics)

audit_columns = [
    "match_date",
    "league",
    "home_team",
    "away_team",
    "football_data_max_home_odds",
    "market_resolution_status",
    "eligible_trade_status",
    "eligible_trade_rows",
    "best_eligible_effective_odds",
]
display(collection.match_summary.reindex(columns=audit_columns).round(6))

not_discovered = collection.match_summary.loc[
    collection.match_summary["market_resolution_status"].eq("not_discovered_in_trade_data"),
    ["match_date", "league", "home_team", "away_team"],
]
no_eligible_execution = collection.match_summary.loc[
    collection.match_summary["eligible_trade_status"].eq("resolved_no_eligible_trades"),
    ["match_date", "league", "home_team", "away_team", "football_data_max_home_odds"],
]

print(f"Matches without market discovery: {len(not_discovered):,}")
if not not_discovered.empty:
    display(not_discovered.reset_index(drop=True))

print(f"Discovered markets without acceptable execution: {len(no_eligible_execution):,}")
if not no_eligible_execution.empty:
    display(no_eligible_execution.reset_index(drop=True))

## 6. Finalize monthly league files

Two files are written for every league with at least one selected fixture:

- `*_match_summary.csv`: one audit row per Football-Data-selected match, including discovery and eligible-trade status;
- `*_eligible_trades.csv`: all canonical direction-correct transaction rows that pass the `MaxH` execution threshold.

Local workbooks and generated CSV files remain excluded from version control. Only code, documentation, tests, and the published frozen parameter snapshot belong in the public repository.

In [ ]:
if SAVE_MONTHLY_OUTPUTS:
    saved_files = save_monthly_outputs_by_league(
        collection,
        selection.selected_fixtures,
        output_dir=OUTPUT_DIR,
        season=selection.season,
        month=selection.month,
        allow_correction_overwrite=ALLOW_CORRECTION_OVERWRITE,
        source=ODDS_SOURCE,
        market_outcome=MARKET_OUTCOME,
    )
    saved_display = saved_files.copy()
    for column in ["match_summary_file", "eligible_trades_file"]:
        saved_display[column] = saved_display[column].map(
            lambda value: str(Path(value).relative_to(PROJECT_ROOT))
        )
    display(saved_display)
else:
    print("SAVE_MONTHLY_OUTPUTS is False; no monthly files were written.")

## 7. Interpretation and limitations

This notebook establishes an auditable monthly execution-opportunity dataset; it does not establish realized strategy returns.

- Football-Data and the frozen snapshot alone determine which matches qualify.
- Dune transaction history is used only after selection and only between the documented Football-Data availability time and kickoff.
- A discovery failure does not prove that no Polymarket market existed; it means that the implemented trade-data search did not resolve one unique home-win market.
- A qualifying historical transaction is evidence that a price and quantity traded. It is not a historical order-book snapshot and does not prove that a hypothetical order would have received that fill.
- Participation, queue position, fees applied to hypothetical execution, open exposure, staking, and settlement belong in the separate forward execution report.

After a completed month is finalized, use its league-level `eligible_trades` files as inputs to the forward execution notebook. Keep the frozen parameters and historical monthly files unchanged unless a correction is explicitly documented.